## GTFS Data Analysis

In [2]:
#Defines data path
data_path=r"D:\GIS-TU Dublin\Year_2\THESIS\DCC_Analysis"

In [3]:
#Imports required modules
import os
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import numpy as np
import matplotlib.pyplot as plt

In [4]:
path=os.getcwd()
print(path)

C:\WINDOWS\system32


In [5]:
#Clipping GTFS to Public Transit Study Area of Dublin City Council (DCC Boundary + 500m buffer)

def clip_gtfs_to_dcc(gtfs_dir, dcc_boundary_file, output_dir):
    """
    Clip GTFS data to Dublin City Council boundary.
    
    Args:
        gtfs_dir: Path to directory containing GTFS files
        dcc_boundary_file: Path to DCC boundary shapefile/GeoJSON
        output_dir: Directory to save clipped GTFS files
    """
    
    # Load DCC boundary
    dcc_boundary = gpd.read_file(dcc_boundary_file)
    
    # Read GTFS files
    stops = pd.read_csv(f"{gtfs_dir}/stops.txt")
    stop_times = pd.read_csv(f"{gtfs_dir}/stop_times.txt")
    trips = pd.read_csv(f"{gtfs_dir}/trips.txt")
    routes = pd.read_csv(f"{gtfs_dir}/routes.txt")
    
    # Convert stops to GeoDataFrame
    geometry = [Point(xy) for xy in zip(stops.stop_lon, stops.stop_lat)]
    stops_gdf = gpd.GeoDataFrame(stops, geometry=geometry, crs="EPSG:4326")
    
    # Transform to same CRS as boundary
    stops_gdf = stops_gdf.to_crs(dcc_boundary.crs)
    
    # Find stops within DCC boundary - using 'predicate' instead of 'op'
    stops_within = gpd.sjoin(stops_gdf, dcc_boundary, how="inner", predicate="within")
    stop_ids_within = set(stops_within.stop_id)
    
    # Filter stops.txt
    stops_filtered = stops[stops.stop_id.isin(stop_ids_within)]
    
    # Filter stop_times.txt
    stop_times_filtered = stop_times[stop_times.stop_id.isin(stop_ids_within)]
    trip_ids_within = set(stop_times_filtered.trip_id)
    
    # Filter trips.txt
    trips_filtered = trips[trips.trip_id.isin(trip_ids_within)]
    route_ids_within = set(trips_filtered.route_id)
    
    # Filter routes.txt
    routes_filtered = routes[routes.route_id.isin(route_ids_within)]
    
    # Save filtered GTFS files
    stops_filtered.to_csv(f"{output_dir}/stops.txt", index=False)
    stop_times_filtered.to_csv(f"{output_dir}/stop_times.txt", index=False)
    trips_filtered.to_csv(f"{output_dir}/trips.txt", index=False)
    routes_filtered.to_csv(f"{output_dir}/routes.txt", index=False)
    
    # Copy other required files (calendar, etc.) without filtering
    # You may need to add additional files depending on your GTFS
    for file in ["calendar.txt", "calendar_dates.txt", "agency.txt", "feed_info.txt"]:
        try:
            pd.read_csv(f"{gtfs_dir}/{file}").to_csv(f"{output_dir}/{file}", index=False)
        except FileNotFoundError:
            pass
# Example usage
if __name__ == "__main__":
    gtfs_dir =data_path + r"\Data_used\GTFS\GTFS_All_August"
    dcc_boundary_file = data_path + r"\Data_used\DCC\PT_Study_Area.shp"  # or .geojson
    output_dir = data_path + r"\Data_used\GTFS\GTFS_Aug_DCC"
    
    clip_gtfs_to_dcc(gtfs_dir, dcc_boundary_file, output_dir)

In [6]:
#Reads GTFS data files of August
stops_Aug=pd.read_csv(data_path + r'\Data_used\GTFS\GTFS_Aug_DCC\stops.txt')
stop_times_Aug=pd.read_csv(data_path + r'\Data_used\GTFS\GTFS_Aug_DCC\stop_times.txt', low_memory=False)
trips_Aug=pd.read_csv(data_path + r'\Data_used\GTFS\GTFS_Aug_DCC\trips.txt')
routes_Aug=pd.read_csv(data_path + r'\Data_used\GTFS\GTFS_Aug_DCC\routes.txt')
calendar_Aug = pd.read_csv(data_path + r'\Data_used\GTFS\GTFS_Aug_DCC\calendar.txt')
calendar_dates_Aug = pd.read_csv(data_path + r'\Data_used\GTFS\GTFS_Aug_DCC\calendar_dates.txt')


### Determining Peak and Off-Peak Hour for Analysis

In [212]:
#Finds Peak and Off-Peak time window based on trips count
# Hours window (inclusive): 08:00–20:59
HOUR_START = 8
HOUR_END   = 20

# Which service_id(s) to include
SERVICE_IDS = {1}

# Output folder
out_dir = os.path.join(data_path, 'GTFS_frequency_outputs')
os.makedirs(out_dir, exist_ok=True)

# HELPERS
def gtfs_time_to_seconds(t):
    """Convert 'HH:MM:SS' (HH may be >=24) to seconds since 00:00."""
    h, m, s = map(int, str(t).split(':'))
    return h*3600 + m*60 + s

def hour_to_window_label(h):
    """Return label like '08:00–09:00' for hour h."""
    return f"{h:02d}:00–{(h+1):02d}:00"

# PREP STOP_TIMES WITH HOUR
st = stop_times_Aug.copy()
st['stop_id'] = st['stop_id'].astype(str)
st['dep_sec'] = st['departure_time'].astype(str).apply(gtfs_time_to_seconds)
st['hour'] = (st['dep_sec'] // 3600).astype(int)

# Keep 08..20
st = st[(st['hour'] >= HOUR_START) & (st['hour'] <= HOUR_END)].copy()

# MERGE TRIPS + ROUTES META
trips_mode_Aug = trips_Aug.merge(routes_Aug[['route_id', 'route_type']], on='route_id', how='left')
mt = st.merge(
    trips_mode_Aug[['trip_id','route_id','service_id','direction_id','route_type']],
    on='trip_id', how='left'
)

# Filter service(s)
mt = mt[mt['service_id'].isin(SERVICE_IDS)].copy()

# ==== DEDUP PER TRIP PER HOUR (avoid overcount from many stops in same hour) ====
mt_unique = mt.drop_duplicates(subset=['trip_id', 'hour'])

# ==== SYSTEM-WIDE HOURLY FREQUENCY ====
hourly = (
    mt_unique.groupby('hour')
    .size()
    .reset_index(name='trips_per_hour')
)

# Ensure all hours appear and add window labels
all_hours = pd.DataFrame({'hour': np.arange(HOUR_START, HOUR_END+1)})
hourly = all_hours.merge(hourly, on='hour', how='left').fillna({'trips_per_hour': 0})
hourly['trips_per_hour'] = hourly['trips_per_hour'].astype(int)
hourly['time_window'] = hourly['hour'].apply(hour_to_window_label)

# Identify peak/off-peak with simple tie-breaks
max_val = hourly['trips_per_hour'].max()
min_val = hourly['trips_per_hour'].min()
peak_candidates = hourly.loc[hourly['trips_per_hour'] == max_val, 'hour'].tolist()
off_candidates  = hourly.loc[hourly['trips_per_hour'] == min_val, 'hour'].tolist()

# Tie-breaks: for peak prefer closest to 8; for off-peak prefer closest to 14
peak_hour = min(peak_candidates, key=lambda h: abs(h-8)) if peak_candidates else None
off_hour  = min(off_candidates,  key=lambda h: abs(h-14)) if off_candidates else None

peak_label = hour_to_window_label(peak_hour) if peak_hour is not None else None
off_label  = hour_to_window_label(off_hour)  if off_hour  is not None else None

# ====BY MODE (route_type) ====
hourly_by_mode = (
    mt_unique.groupby(['hour','route_type'])
    .size().reset_index(name='trips_per_hour')
)
hourly_by_mode['time_window'] = hourly_by_mode['hour'].apply(hour_to_window_label)
hourly_by_mode_pivot = (
    hourly_by_mode.pivot(index='time_window', columns='route_type', values='trips_per_hour')
    .reindex(hourly['time_window'])
    .fillna(0).astype(int)
)
hourly_by_mode_pivot.index.name = 'time_window'

# ==== PRINT SUMMARY ====
print("\n=== Hourly trips (distinct trips/hour, Aug) ===")
print(hourly[['time_window','trips_per_hour']].to_string(index=False))

if peak_label is not None:
    peak_val = int(hourly.loc[hourly['time_window']==peak_label, 'trips_per_hour'].iloc[0])
    print(f"\nPEAK window  = {peak_label} with {peak_val} trips")
if off_label is not None:
    off_val = int(hourly.loc[hourly['time_window']==off_label, 'trips_per_hour'].iloc[0])
    print(f"OFF-PEAK window = {off_label} with {off_val} trips")

print("\n=== Hourly trips by mode (route_type) ===")
print(hourly_by_mode_pivot)

# ==== SAVE CSVs ====
hourly[['time_window','trips_per_hour']].to_csv(os.path.join(out_dir, 'hourly_trips_Aug_08to20.csv'), index=False)
hourly_by_mode_pivot.to_csv(os.path.join(out_dir, 'hourly_trips_by_mode_Aug_08to20.csv'))

# ==== PLOT LINE CHART ====
plt.figure(figsize=(10,5))
plt.plot(hourly['time_window'], hourly['trips_per_hour'], marker='o', linewidth=2)
plt.title("Scheduled Public Transport Trips per Hour (Aug, Weekday)")
plt.xlabel("Time Window")
plt.ylabel("Trips per Hour (distinct trips)")
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)

# Marks peak & off-peak
if peak_label is not None:
    x_idx = hourly.index[hourly['time_window'] == peak_label][0]
    plt.scatter([hourly['time_window'].iloc[x_idx]], [hourly['trips_per_hour'].iloc[x_idx]], s=80)
    plt.annotate("Peak", (x_idx, hourly['trips_per_hour'].iloc[x_idx]), xytext=(0,8),
                 textcoords="offset points", ha='center')
if off_label is not None:
    x_idx = hourly.index[hourly['time_window'] == off_label][0]
    plt.scatter([hourly['time_window'].iloc[x_idx]], [hourly['trips_per_hour'].iloc[x_idx]], s=80)
    plt.annotate("Off-peak", (x_idx, hourly['trips_per_hour'].iloc[x_idx]), xytext=(0,8),
                 textcoords="offset points", ha='center')

plt.tight_layout()
plt.savefig(os.path.join(out_dir, "Hourly_trips_Aug_linechart.png"), dpi=300)
plt.show()



=== Hourly trips (distinct trips/hour, Aug) ===
time_window  trips_per_hour
08:00–09:00              33
09:00–10:00              16
10:00–11:00               7
11:00–12:00               5
12:00–13:00               6
13:00–14:00               4
14:00–15:00               4
15:00–16:00               6
16:00–17:00              20
17:00–18:00              28
18:00–19:00              21
19:00–20:00               5
20:00–21:00               5

PEAK window  = 08:00–09:00 with 33 trips
OFF-PEAK window = 14:00–15:00 with 4 trips

=== Hourly trips by mode (route_type) ===
route_type    3
time_window    
08:00–09:00  33
09:00–10:00  16
10:00–11:00   7
11:00–12:00   5
12:00–13:00   6
13:00–14:00   4
14:00–15:00   4
15:00–16:00   6
16:00–17:00  20
17:00–18:00  28
18:00–19:00  21
19:00–20:00   5
20:00–21:00   5


In [8]:
#Reads stops.txt file
stops_Aug.head()
print(f'Aug Stop : {stops_Aug.shape}')

Aug Stop : (2250, 10)


In [9]:
# Show full details for service_id = 1
calendar_Aug[calendar_Aug['service_id'] == 1]

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1,1,1,1,1,1,0,0,20250829,20251031


In [10]:
#Reads routes.txt GTFS file
routes_Aug.head()

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
0,4194_46238,7778230,164,Sydney Parade DART Station - UCD,NaN,3,NaN,NaN,NaN
1,4246_40618,7778066,501X,"Swords, Pavilions Shopping Centre - Dublin, Ma...",NaN,3,NaN,NaN,NaN
2,4246_40622,7778066,504,"Rathingle, Boroimhe Estate - Dublin, Marlborou...",NaN,3,NaN,NaN,NaN
3,4246_59686,7778066,500N,"Dublin, Eden Quay - Swords Manor Brackenstown ...",NaN,3,NaN,NaN,NaN
4,4246_63368,7778066,500,Swords - Dublin City,NaN,3,NaN,NaN,NaN


In [11]:
routes_Aug['route_type'].value_counts().reset_index()

,index,route_type
0,3,242
1,2,14
2,0,2


In [12]:
#Reads GTFS Aug stop_times.txt file
print(f'Aug Stop Times: {stop_times_Aug.shape}')
stop_times_Aug.head()

Aug Stop Times: (1224067, 9)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,timepoint
0,4194_1,08:00:00,08:00:00,8220DB002087,1,Stillorgan Road,0,1,1
1,4194_1,08:15:00,08:15:00,825000175,2,NaN,1,0,1
2,4194_10,18:00:00,18:00:00,825000354,1,"Ailesbury Road, stop 2087",0,1,1
3,4194_10,18:10:00,18:10:00,8220DB002087,2,NaN,1,0,1
4,4194_2,08:30:00,08:30:00,8220DB002087,1,Stillorgan Road,0,1,1


In [13]:
#Reads GTFS Aug trips.txt file
print(f'trips_Aug: {trips_Aug.shape}')
trips_Aug.head()

trips_Aug: (61741, 8)


,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id
0,4194_46238,31,4194_1,Stillorgan Road,1.Mo-Fr.3-164-y11-3.,0,4194_7778232_Txc100000,4194_1
1,4194_46238,31,4194_10,"Ailesbury Road, stop 2087",10.Mo-Fr.3-164-y11-3,1,4194_7778232_Txc100001,4194_2
2,4194_46238,31,4194_2,Stillorgan Road,2.Mo-Fr.3-164-y11-3.,0,4194_7778232_Txc100002,4194_1
3,4194_46238,31,4194_3,Stillorgan Road,3.Mo-Fr.3-164-y11-3.,0,4194_7778232_Txc100003,4194_1
4,4194_46238,31,4194_4,Stillorgan Road,4.Mo-Fr.3-164-y11-3.,0,4194_7778232_Txc100004,4194_1


In [14]:
trips_Aug['service_id'].unique()

array([ 31,   1,   8,   2,  32,  33,   5,   4,  36,   3,  43,  44,  45,
        46,  47,  49,  50,  48,  51,  52,  55,  56,  34,  58,  59,  28,
        60,  61,  62,  72,  73,  74,  75,  76,  78,  79,  77,  80,  81,
        83,  22,  84,  85,  86,  35,  23,  25,  87,  26,  94,  95,  96,
        97,  98,  89,  90,  91,  92,  93,  88,  99, 100, 101,  41, 102,
       103, 212, 214, 218, 215, 216, 217, 219, 220, 221, 237, 238, 239,
       240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252,
       253, 254, 255, 226, 227, 228, 229, 231, 230, 232, 233, 234, 235,
       236, 176, 123, 260, 148, 261,  39, 225, 223, 256, 296, 297, 298,
       299, 300, 166, 304, 306, 303, 302, 309, 305, 316, 343, 308, 333,
       104, 344, 345, 311, 317, 321, 174, 359, 346, 349, 348, 350, 351,
       352, 353, 354, 355, 356, 357, 358, 347, 360, 361,  15, 314, 362,
       307, 310, 315, 318, 319, 320, 322, 323, 324, 326, 327, 328, 329,
       330, 325, 331, 312, 332, 334, 335, 336, 337, 338, 340, 34

In [15]:
#Merges trips.txt file with routes.txt file
trips_mode_Aug = trips_Aug.merge(routes_Aug[['route_id','route_type']], on='route_id', how ='left')
trips_mode_Aug.head()

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,route_type
0,4194_46238,31,4194_1,Stillorgan Road,1.Mo-Fr.3-164-y11-3.,0,4194_7778232_Txc100000,4194_1,3
1,4194_46238,31,4194_10,"Ailesbury Road, stop 2087",10.Mo-Fr.3-164-y11-3,1,4194_7778232_Txc100001,4194_2,3
2,4194_46238,31,4194_2,Stillorgan Road,2.Mo-Fr.3-164-y11-3.,0,4194_7778232_Txc100002,4194_1,3
3,4194_46238,31,4194_3,Stillorgan Road,3.Mo-Fr.3-164-y11-3.,0,4194_7778232_Txc100003,4194_1,3
4,4194_46238,31,4194_4,Stillorgan Road,4.Mo-Fr.3-164-y11-3.,0,4194_7778232_Txc100004,4194_1,3


In [18]:
# --- choose an analysis date  ---
date_str = "2025-09-30"
date_int = int(date_str.replace("-", ""))
weekday_col = pd.Timestamp(date_str).day_name().lower()  # 'monday'..'sunday'

# --- build the set of active service_ids for that date (calendar + exceptions) ---
cal = calendar_Aug.copy()
cal.columns = [c.lower() for c in cal.columns]
for c in ["start_date","end_date"]:
    if cal[c].dtype.kind not in "iu":
        cal[c] = cal[c].astype(int)

base_mask = (cal[weekday_col].eq(1) & (cal["start_date"] <= date_int) & (cal["end_date"] >= date_int))
base_services = set(cal.loc[base_mask, "service_id"])

cd = calendar_dates_Aug.copy()
cd.columns = [c.lower() for c in cd.columns]
if cd["date"].dtype.kind not in "iu":
    cd["date"] = cd["date"].astype(int)

adds    = set(cd.loc[(cd["date"] == date_int) & (cd["exception_type"] == 1), "service_id"])
removes = set(cd.loc[(cd["date"] == date_int) & (cd["exception_type"] == 2), "service_id"])

active_services = (base_services - removes) | adds

In [20]:
#Filtering the trips for service_id=1 and morning-peak 8am -10 am
filtered_trips = trips_mode_Aug[trips_mode_Aug['service_id'].isin(active_services)]
# Remove quotation marks from the departure_time column
stop_times_Aug['departure_time'] = stop_times_Aug['departure_time'].str.replace('"', '').str.replace("'", "")

# Copy stop_times to avoid modifying original
st = stop_times_Aug.copy()

# Clean departure_time column
st['departure_time'] = st['departure_time'].astype(str)
st = st[st['departure_time'].str.match(r'^\d{1,2}:\d{2}:\d{2}$', na=False)]

# Convert HH:MM:SS to seconds since midnight (handles HH >= 24)
def hms_to_sec(s):
    h, m, sec = map(int, s.split(':'))
    return h * 3600 + m * 60 + sec

st['dep_sec'] = st['departure_time'].apply(hms_to_sec)

# Filter for time windows
mp_start, mp_end = 8*3600, 9*3600    # 08:00–09:00
op_start, op_end = 13*3600, 14*3600  # 13:00–14:00

filtered_stops_mp = st[(st['dep_sec'] >= mp_start) & (st['dep_sec'] <= mp_end)]
filtered_stops_op = st[(st['dep_sec'] >= op_start) & (st['dep_sec'] <= op_end)]

# filtered_stops_mp = stop_times_Aug[(stop_times_Aug['departure_time'] >= '08:00:00') & (stop_times_Aug['departure_time'] <= '9:00:00')]
# filtered_stops_op = stop_times_Aug[(stop_times_Aug['departure_time'] >= '13:00:00') & (stop_times_Aug['departure_time'] <= '14:00:00')]
print(f'Filtered Trips: {filtered_trips.shape[0]}')
print(f'Filtered Stops: {filtered_stops_mp.shape[0]}')
print(f'Filtered Stops: {filtered_stops_op.shape[0]}')

Filtered Trips: 13192
Filtered Stops: 65768
Filtered Stops: 72472


In [21]:
# Finds the true min/max within the 08:00–09:00 window?
print("min dep_sec:", filtered_stops_mp['dep_sec'].min())  # expect 28800 (08:00:00)
print("max dep_sec:", filtered_stops_mp['dep_sec'].max())  # expect < 32400 (09:00:00)


min dep_sec: 28800
max dep_sec: 32400


In [22]:
#Checks what times are actually in data
print("Unique departure times sample:")
print(stop_times_Aug['departure_time'].unique()[:10])  # Look for times > 24:00:00

Unique departure times sample:
['08:00:00' '08:15:00' '18:00:00' '18:10:00' '08:30:00' '08:45:00'
 '09:00:00' '09:15:00' '09:30:00' '09:45:00']


In [23]:
filtered_stops_mp.head(2)

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,timepoint,dep_sec
0,4194_1,08:00:00,08:00:00,8220DB002087,1,Stillorgan Road,0,1,1,28800
1,4194_1,08:15:00,08:15:00,825000175,2,NaN,1,0,1,29700


In [24]:
filtered_stops_mp.tail(2)

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,timepoint,dep_sec
1223897,4876_97,08:28:05,08:28:05,8220GA00276,16,NaN,0,0,1,30485
1223898,4876_97,08:29:43,08:29:43,8220GA00279,17,NaN,0,0,1,30583


In [25]:
filtered_stops_op.head(2)

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,timepoint,dep_sec
64,4246_106,13:00:00,13:00:00,8220DB007359,1,Ormond Avenue,0,1,1,46800
65,4246_106,13:01:00,13:01:00,8220DB002498,2,NaN,0,0,1,46860


In [26]:
filtered_stops_op.tail(2)

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,timepoint,dep_sec
1216099,4876_5309,13:59:45,13:59:45,8220GA00364,9,NaN,0,0,1,50385
1216146,4876_5311,13:59:46,13:59:46,8220GA00357,7,NaN,0,0,1,50386


In [27]:
#Merging filtered trips to one data
merged_trips_stops_Aug_mp = filtered_trips.merge(filtered_stops_mp, on='trip_id', how='inner')
merged_trips_stops_Aug_op = filtered_trips.merge(filtered_stops_op, on='trip_id', how='inner')
print(f'Merged trips stops mp: {merged_trips_stops_Aug_mp.shape}')
print(f'Merged trips stops op: {merged_trips_stops_Aug_op.shape}')

Merged trips stops mp: (22204, 18)
Merged trips stops op: (18238, 18)


In [28]:
# Calculate frequencies for morning peak
freq_MP = (
    merged_trips_stops_Aug_mp
    .groupby(['stop_id', 'route_type'])
    .size()
    .reset_index(name='freq_MP')
)
# Calculate frequencies for off-peak
freq_OP = (
    merged_trips_stops_Aug_op
    .groupby(['stop_id', 'route_type'])
    .size()
    .reset_index(name='freq_OP')
)

In [29]:
# Merge both frequency tables
freq_combined = freq_MP.merge(freq_OP, on=['stop_id', 'route_type'], how='outer').fillna(0)
freq_combined['freq_MP'] = freq_combined['freq_MP'].astype(int)
freq_combined['freq_OP'] = freq_combined['freq_OP'].astype(int)


In [30]:
# Calculate changes
freq_combined['freq_change'] = freq_combined['freq_MP'] - freq_combined['freq_OP']
freq_combined['pct_change'] = (
    (freq_combined['freq_MP'] - freq_combined['freq_OP']) / 
    freq_combined['freq_OP'].replace(0, 1) * 100  # Avoid division by zero
)
freq_combined['pct_change'] = freq_combined['pct_change'].round(2)

In [31]:
freq_combined.head()

,stop_id,route_type,freq_MP,freq_OP,freq_change,pct_change
0,8220000002,3,1,0,1,100.0
1,8220000005,3,2,1,1,100.0
2,8220000150,3,3,2,1,50.0
3,8220000357,3,1,1,0,0.0
4,8220000372,3,16,2,14,700.0


In [32]:
freq_combined.shape

(2192, 6)

In [33]:
# Add stop information
stops_freq = freq_combined.merge(
    stops_Aug[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
    on='stop_id', how='left'
)
stops_freq.head()

,stop_id,route_type,freq_MP,freq_OP,freq_change,pct_change,stop_name,stop_lat,stop_lon
0,8220000002,3,1,0,1,100.0,Sheriff Street Upper,53.349883,-6.230232
1,8220000005,3,2,1,1,100.0,Park West Road,53.332343,-6.363435
2,8220000150,3,3,2,1,50.0,Abbey Street,53.348562,-6.258961
3,8220000357,3,1,1,0,0.0,George's Quay,53.347372,-6.252596
4,8220000372,3,16,2,14,700.0,East Wall Road,53.350791,-6.226003


In [ ]:
#Converts the crs from wgs84 to ITM and saves the shapefile to desired folder
stops_freq = gpd.GeoDataFrame(
    stops_freq,
    geometry=gpd.points_from_xy(stops_freq['stop_lon'], stops_freq['stop_lat']),
    crs="EPSG:4326"  
stops_freq_itm = stops_freq.to_crs(epsg=2157)
stops_freq_itm.to_file(data_path + r"\GTFS_frequency_outputs\all_stops_freq_ITM.shp", driver="ESRI Shapefile", encoding="utf-8")
